In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
MODE = "light"

In [ ]:
import os
import sys
import django

# Setup Django environment
# Adjust the path to point to the directory containing manage.py
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../hiccup_ide")))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hiccup_ide.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

In [ ]:
from neural_data.models import *
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
    show_72,
    show_72_list,
    scatter_plot_1d,
    mk_rect_on_ax,
    get_receptive,
    otsu_threshold,
    explain_variance_with_pca,
)
from tqdm import tqdm
from pt_to_api.contribs.v1 import (
    show_input_patch_and_kernel_placement_for_poi_using_raw_params as SIP,
)
import seaborn as sns
import numpy as np
from sklearn.decomposition import MiniBatchDictionaryLearning, DictionaryLearning
from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment
import pandas as pd

from neural_data.utils import (
    get_full_conv_kernel_at_coordinate,
    get_saliency_map_ids_and_patches,
    get_full_activations_of_layer,
)
from collections import defaultdict
import itertools
from torch import nn
from torch import optim


In [ ]:
def show_gram(W):
    W_norm = W / (W.norm(dim=0, keepdim=True) + 1e-8)
    gram = W_norm.T @ W_norm  # (n_components, n_components)
    plt.imshow(gram, cmap="gray")
    plt.show()
    return gram

In [ ]:
class GroupingAutoencoderFixedSigma(nn.Module):
    def __init__(self, input_dim, n_components):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, n_components, bias=True),
        )
        self.decoder = nn.Linear(n_components, input_dim, bias=False)

    def forward(self, x):
        codes = self.encoder(x)
        recon = self.decoder(codes)
        return recon, codes

    def loss_fn(self, x, recons, codes, alpha, sigma_x, sigma_s, sigma_0):
        recons_loss = self.recon_loss(x, recons, sigma_x)

        codes_loss = self.codes_loss(codes, sigma_s)

        weights_loss = self.weights_loss(alpha, sigma_0)

        return recons_loss, codes_loss, weights_loss

    def codes_loss(self, codes, sigma_s):
        return self.gauss_loss(codes, 0) / (sigma_s*sigma_s)

    def recon_loss(self, x, recons, sigma_x):
        return self.gauss_loss(x, recons) / (sigma_x*sigma_x)

    def cauchy_loss(self, x, cauchy_gamma):
        return torch.log(1 + (x*x) / (cauchy_gamma*cauchy_gamma)).sum()

    def cauchy_codes_loss(self, codes, cauchy_gamma):
        return torch.log(1 + (codes*codes) / (cauchy_gamma*cauchy_gamma)).sum(1).mean()

    def gauss_loss(self, x, mean):
        loss = ((x - mean) ** 2)
        # sum the loss inside one example, send back mean across examples for the batch
        return torch.sum(loss, 1).mean()

    def l1_loss(self, x, mean):
        loss = (x - mean).abs()
        return torch.sum(loss, 1).mean()

    def weights_loss(self, alpha, sigma_0):
        W = self.decoder.weight
        W_sq = W ** 2  # (C, K)
        cumsum = torch.cumsum(W_sq, dim=1)          # (C, K), cumsum[c,k] = sum W[c,0..k]^2
        phi = alpha * torch.roll(cumsum, 1, dims=1) + 1  # (C, K)
        phi[:, 0] = 1  # k=0: phi_weight(W, c, -1, alpha) = alpha*0 + 1
        comp1 = (W_sq * phi) / (sigma_0*sigma_0)
        comp2 = -torch.log(phi)
        # we just send the sum, this is per example loss btw
        # the others are sending mean so all good
        return (comp1 + comp2).sum()

    def weights_loss_cycled(self, alpha, sigma_0, chunk_size=64):
        W = self.decoder.weight          # (C, K)
        C, K = W.shape

        idx = (torch.arange(K, device=W.device).unsqueeze(0) +
            torch.arange(K, device=W.device).unsqueeze(1)) % K   # (K, K)

        shift_losses = []

        for start in range(0, K, chunk_size):
            idx_chunk = idx[start:start + chunk_size]   # (chunk, K)

            W_chunk = W[:, idx_chunk]                   # (C, chunk, K)
            W_chunk = W_chunk.permute(1, 0, 2)          # (chunk, C, K)

            W_sq = W_chunk ** 2

            cumsum = torch.cumsum(W_sq, dim=2)
            phi = alpha * torch.roll(cumsum, 1, dims=2) + 1
            phi[:, :, 0] = 1

            comp1 = (W_sq * phi) / (sigma_0 * sigma_0)
            comp2 = -torch.log(phi)

            shift_losses.append((comp1 + comp2).sum(dim=(1, 2)))  # (chunk,)

        return torch.cat(shift_losses).mean()

    def masked_recon_loss(self, x, recons, sigma_x):
        mask = (x != 0).float()
        loss = ((x - recons) ** 2) * mask
        return torch.sum(loss, 1).mean() / (sigma_x**2)

In [ ]:
def make_dim_partition(patch_dim, n_components, seed=42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_components]) for i in range(n_components)]

def generate_synthetic_patches(patch_dim=72, n_components=10, k=3, n_samples=1000, noise_std=0.01, seed=42):
    rng = np.random.RandomState(seed)
    
    dim_partition = make_dim_partition(patch_dim, n_components, seed)
    
    # ground truth atoms, nonzero only on owned dims
    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        W_true[i, dims] = rng.randn(len(dims))
    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)
    
    # each sample uses exactly k atoms
    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        idx = rng.choice(n_components, k, replace=False)
        codes_true[i, idx] = rng.randn(k)
    
    X = codes_true @ W_true
    X += rng.randn(*X.shape) * noise_std
    
    return X, W_true, codes_true, dim_partition

def evaluate_recovery(W_learned, W_true, threshold=0.95):
    """
    W_learned: (n_atoms, patch_dim)
    W_true: (n_atoms, patch_dim)
    
    for each true atom, finds the best matching learned atom by cosine similarity
    returns fraction of true atoms recovered above threshold
    """
    W_l = W_learned / (np.linalg.norm(W_learned, axis=1, keepdims=True) + 1e-8)
    W_t = W_true / (np.linalg.norm(W_true, axis=1, keepdims=True) + 1e-8)
    
    sim = np.abs(W_l @ W_t.T)  # (n_atoms, n_atoms), abs because sign is arbitrary
    best_match = sim.max(axis=0)  # for each true atom, best cosine with any learned atom
    
    recovered = (best_match >= threshold).mean()
    print(f"Mean best cosine similarity: {best_match.mean():.4f}")
    print(f"Fraction recovered (>{threshold}): {recovered:.4f}")
    return best_match, recovered


def _match_atoms(D1: np.ndarray, D2: np.ndarray):
    """
    Match atoms of D1 to atoms of D2 using the Hungarian algorithm
    on cosine distances. Assumes square dictionaries (same n_components).

    D1, D2: shape (n_components, n_features) — sklearn's components_ layout.

    Returns:
        row_ind, col_ind: matched index arrays
        matched_similarities: per-pair cosine similarities
        mean_sim: mean cosine similarity across matched pairs
    """
    # Guard against dead atoms (zero-norm rows produce NaN cosine distances)
    norms_1 = np.linalg.norm(D1, axis=1, keepdims=True)
    norms_2 = np.linalg.norm(D2, axis=1, keepdims=True)
    if np.any(norms_1 == 0) or np.any(norms_2 == 0):
        raise ValueError(
            "One or more atoms have zero norm. "
            "Remove or replace dead atoms before matching."
        )

    # cost = cosine_distances(D1, D2)          # shape (n_components, n_components), values in [0, 2]
    cost = np.abs(cosine_similarity(D1, D2))
    cost = 1 - cost

    row_ind, col_ind = linear_sum_assignment(cost)
    matched_similarities = 1 - cost[row_ind, col_ind]
    mean_sim = float(matched_similarities.mean())
    return row_ind, col_ind, matched_similarities, mean_sim


def hungarian_match(all_components: list[np.ndarray]):
    """
    Pairwise similarity matching across runs using the Hungarian algorithm.

    Args:
        all_components: list of arrays, each shape (n_components, n_features).
                        All arrays must have the same shape.

    Returns:
        upper: 1-D array of pairwise similarities for all unique pairs
        stability_score: mean of upper
        best_run_idx: index of the run most similar to all others
        pairwise_sims: (n_runs, n_runs) symmetric similarity matrix, diagonal = 1
    """
    n_runs = len(all_components)

    if n_runs < 2:
        raise ValueError("Need at least 2 runs to compute pairwise similarity.")

    shapes = [d.shape for d in all_components]
    if len(set(shapes)) != 1:
        raise ValueError(
            f"All dictionaries must have the same shape. Got: {shapes}"
        )

    pairwise_sims = np.ones((n_runs, n_runs))
    for i in range(n_runs):
        for j in range(i + 1, n_runs):
            _, _, _, mean_sim = _match_atoms(all_components[i], all_components[j])
            pairwise_sims[i, j] = mean_sim
            pairwise_sims[j, i] = mean_sim

    upper = pairwise_sims[np.triu_indices(n_runs, k=1)]
    stability_score = float(upper.mean())

    # Exclude self-similarity (diagonal=1) when ranking runs
    np.fill_diagonal(pairwise_sims, 0)
    mean_sim_per_run = pairwise_sims.sum(axis=1) / (n_runs - 1)
    best_run_idx = int(np.argmax(mean_sim_per_run))
    np.fill_diagonal(pairwise_sims, 1)  # restore diagonal

    return upper, stability_score, best_run_idx, pairwise_sims

In [ ]:
def make_subset_atoms(W_true, dim_partition, ratios, seed=42):
    """
    Creates subset atoms, each borrowing dims from a single randomly chosen parent atom.

    Args:
        W_true:        (n_components, patch_dim) ground truth atoms
        dim_partition: list of lists, dims owned by each atom
        ratios:        list of floats, per-sample activation probability for each subset atom
                       (implicitly defines how many subset atoms to add)
        seed:          random seed

    Returns:
        W_subset:      (len(ratios), patch_dim) subset atom weight matrix
        parents:       list of int, parent atom index for each subset atom
        ratios:        echo back for use in sampling
    """
    rng = np.random.RandomState(seed)
    n_components = len(dim_partition)
    patch_dim = W_true.shape[1]
    n_subset = len(ratios)

    assert n_subset <= n_components, "More subset atoms than available parent atoms"

    # each subset atom gets a distinct parent
    parents = list(rng.choice(n_components, n_subset, replace=False))

    W_subset = np.zeros((n_subset, patch_dim))
    for i, parent in enumerate(parents):
        parent_dims = dim_partition[parent]
        # random non-empty subset of parent's dims
        subset_size = rng.randint(1, len(parent_dims))
        chosen_dims = rng.choice(parent_dims, subset_size, replace=False)
        W_subset[i, chosen_dims] = W_true[parent, chosen_dims]

    # re-normalize
    norms = np.linalg.norm(W_subset, axis=1, keepdims=True)
    W_subset /= np.where(norms > 0, norms, 1)

    return W_subset, parents, ratios


def sample_with_subset_atoms(W_true, W_subset, parents, ratios, k, n_samples, noise_std=0.01, seed=42):
    """
    Generates samples using both base and subset atoms.

    For each subset atom s with parent p:
      - with probability ratios[s]: s fires (additively), p is excluded from the k active base atoms
      - otherwise: neither s fires, p is available as normal

    Total active atoms per sample = k (base, minus blocked parents) + (fired subset atoms)
    """
    rng = np.random.RandomState(seed)
    n_components = W_true.shape[0]
    n_subset = len(ratios)
    patch_dim = W_true.shape[1]
    W_all = np.vstack([W_true, W_subset])  # (n_components + n_subset, patch_dim)

    X = np.zeros((n_samples, patch_dim))
    codes = np.zeros((n_samples, n_components + n_subset))

    for i in range(n_samples):
        fired_subsets = []
        blocked_parents = set()

        for s, (parent, ratio) in enumerate(zip(parents, ratios)):
            if rng.rand() < ratio:
                fired_subsets.append(s)
                blocked_parents.add(parent)

        available = [j for j in range(n_components) if j not in blocked_parents]
        chosen_base = list(rng.choice(available, min(k, len(available)), replace=False))

        active = chosen_base + [n_components + s for s in fired_subsets]
        coeffs = rng.randn(len(active))

        for idx, coeff in zip(active, coeffs):
            codes[i, idx] = coeff
            X[i] += coeff * W_all[idx]

    X += rng.randn(*X.shape) * noise_std
    return X, codes, W_all

In [ ]:

def get_alpha(epoch, total_epochs, alpha_start=0.0, alpha_end=1.0):
    # do the last 25% with max_alpha
    epoch_max = int(total_epochs * 0.30)
    return alpha_start + (alpha_end - alpha_start) * (epoch / epoch_max)

def train_grouping_autoencoder_fixed_sigma(
    X, n_components, max_alpha=5000, sigma_x=0.1, sigma_s=1, sigma_0=1, lr=1e-3, epochs=2000, batch_size=256, warmup_epochs=5, w_avg_num_seeds=5,
):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: lorentzian sigma shrinker parameter
    sigma_x: std of noise in the data after modelling it as a WS
    sigma_0: std of W, useful to keepn very near 0
    sigma_s: cauchy gamma for cauchy penalty on the encoder (useful for sparse weights)
       the name is sigma_s, cuz it was used as a gaussian prior (L2) on encoder weights, for data which is not sparse
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape
 
    model = GroupingAutoencoderFixedSigma(input_dim, n_components)
    # torch.nn.init.normal_(model.decoder.weight, 0, 1)

    # svd, might add back again later
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    model.decoder.weight.data = torch.tensor(Vt[:n_components].T, dtype=torch.float32)
    optimizer = optim.Adam(model.parameters(), lr=lr)


    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]

        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i:i+batch_size]

            # first do on recons, let everything flow, make LR for weight 0
            recon, codes = model(batch)
            recon_loss = model.recon_loss(batch, recon, sigma_x)
            # alpha = get_alpha(epoch, epochs, 100, max_alpha)
            alpha = max_alpha
            weight_loss = model.weights_loss_cycled(alpha, sigma_0)
            code_loss = model.gauss_loss(model.encoder[0].weight, 0) / (sigma_s*sigma_s)

            loss = recon_loss + weight_loss + code_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()



        if epoch % 200 == 0:
            msg = f"epoch {epoch:4d} | recon_loss {recon_loss:.4f}"
            if weight_loss is not None:
                msg += (f" weight_loss {weight_loss:.4f} codes_loss {code_loss:.4f}")
            # msg += f" reg_weight: {reg_weight}"
            print(msg)


    # final codes and reconstruction
    with torch.no_grad():
        recon, codes = model(X_t)
 
    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )


def train_baseline(
    X, n_components, lr=1e-3, epochs=2000, batch_size=256, sigma_x=1
):
    # simple linear model without any non linearities
    # for loss checking
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape
 
    model = GroupingAutoencoderFixedSigma(input_dim, n_components)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]
        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i:i+batch_size]
            recon, codes = model(batch)
            recon_loss = model.recon_loss(batch, recon, sigma_x)
            loss = recon_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if epoch % 200 == 0:
            print(f"finetune epoch {epoch:4d} | recon_loss {recon_loss:.4f}")

    with torch.no_grad():
        recon, codes = model(X_t)
 
    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )

In [ ]:
def train_autoencoder_coordinate_descent(
    X, n_components, max_alpha=5000, sigma_x=0.1, sigma_s=1, sigma_0=1, lr=1e-3, epochs=2000, batch_size=256
):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: lorentzian sigma shrinker parameter
    sigma_x: std of noise in the data after modelling it as a WS
    sigma_0: std of W, useful to keepn very near 0
    sigma_s: cauchy gamma for cauchy penalty on the encoder (useful for sparse weights)
       the name is sigma_s, cuz it was used as a gaussian prior (L2) on encoder weights, for data which is not sparse
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape
 
    model = GroupingAutoencoderFixedSigma(input_dim, n_components)
    # torch.nn.init.normal_(model.decoder.weight, 0, 1)

    # svd, might add back again later
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    model.decoder.weight.data = torch.tensor(Vt[:n_components].T, dtype=torch.float32)


    # not sure how well gradient descent would work though hmmmm
    # hmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmm
    enc_optim = optim.Adam(model.encoder.parameters(), lr=lr)
    dec_optim = optim.Adam(model.decoder.parameters(), lr=lr)


    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]

        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i:i+batch_size]

            # first do on recons, let everything flow, make LR for weight 0
            recon, codes = model(batch)
            recon_loss = model.recon_loss(batch, recon, sigma_x)
            # alpha = get_alpha(epoch, epochs, 100, max_alpha)
            alpha = max_alpha
            weight_loss = model.weights_loss(alpha, sigma_0)
            code_loss = model.gauss_loss(model.encoder[0].weight, 0) / (sigma_s*sigma_s)

            loss = recon_loss + weight_loss + code_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()



        if epoch % 200 == 0:
            msg = f"epoch {epoch:4d} | recon_loss {recon_loss:.4f}"
            if weight_loss is not None:
                msg += (f" weight_loss {weight_loss:.4f} codes_loss {code_loss:.4f}")
            # msg += f" reg_weight: {reg_weight}"
            print(msg)


    # final codes and reconstruction
    with torch.no_grad():
        recon, codes = model(X_t)
 
    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )


def train_baseline(
    X, n_components, lr=1e-3, epochs=2000, batch_size=256, sigma_x=1
):
    # simple linear model without any non linearities
    # for loss checking
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape
 
    model = GroupingAutoencoderFixedSigma(input_dim, n_components)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]
        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i:i+batch_size]
            recon, codes = model(batch)
            recon_loss = model.recon_loss(batch, recon, sigma_x)
            loss = recon_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if epoch % 200 == 0:
            print(f"finetune epoch {epoch:4d} | recon_loss {recon_loss:.4f}")

    with torch.no_grad():
        recon, codes = model(X_t)
 
    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )

In [ ]:
N_COMPS = 3
N_DIM = 9
X, W_true, codes_true, dim_partition = generate_synthetic_patches(N_DIM, N_COMPS)

In [ ]:
# in some seeds, recon loss does not descend in the beginnign
# its useful to let it descend a bit in the beginning it seems, a few epochs maybe?
# is sparse ness a problem with the other dataset?
# if the data is already sparse, what happens to our weights regularisation?
all_components, all_recons, all_codes = [], [], []

code_maker_std = 1
sigma_0_std = np.std(X) / 2
sigma_noise = sigma_0_std / 100

for seed in [10]:
    print(f"#### seed run: {seed} #########")
    torch.manual_seed(seed)
    model, codes, components, recons = train_grouping_autoencoder_fixed_sigma(
        X, N_COMPS, epochs=2000, sigma_0=sigma_0_std, sigma_s=code_maker_std, sigma_x=sigma_noise, max_alpha=10000, batch_size=256
    )
    all_components.append(components)
    all_recons.append(recons)
    all_codes.append(codes)

In [ ]:
i = 3
print(codes[16])
S([recons[i].reshape(3,3), X[i].reshape(3,3)], mode=MODE)

In [ ]:
components.shape, W_true
sims = np.abs(cosine_similarity(components, W_true))
pairs = []
for i in range(len(components)):
    j = np.argmax(sims[i])
    pairs.append((i, j, sims[i][j]))
for (i,j,score) in pairs:
    print(i, score)
    S([components[i].reshape(3,3), W_true[j].reshape(3,3)], mode=MODE)
    plt.show()

In [ ]:
from sklearn import decomposition
# Global centering (focus on one feature, centering all samples)
means = X.mean(axis=0)
lin_pw_centered = X - means
# lin_pw_centered = X

# Local centering (focus on one sample, centering all features)
# lin_pw_centered -= lin_pw_centered.mean(axis=1).reshape(X.shape[0], -1)
ica_estimator = decomposition.FastICA(
    n_components=N_COMPS, max_iter=2000, whiten="unit-variance", tol=15e-5, fun="exp"
)
ica_estimator.fit(lin_pw_centered)

ica_codes = ica_estimator.transform(lin_pw_centered)
ica_recon = ica_estimator.inverse_transform(ica_codes)
ica_comps = ica_estimator.components_

In [ ]:
S([lin_pw_centered[0].reshape(3,3), X[0].reshape(3,3)], mode=MODE)

In [ ]:
S([c.reshape(3,3) for c in ica_comps], ncols=3, mode=MODE)

In [ ]:
W_true

In [ ]:
S([c.reshape(3,3) for c in W_true], ncols=3, mode=MODE)

In [ ]:
W_true[2]

In [ ]:
ica_comps

In [ ]:
upper, stability_score, _, pairwise_sims = hungarian_match(all_components)
print(stability_score)
plt.imshow(pairwise_sims, cmap="gray", vmin=0, vmax=1)
plt.show()

In [ ]:
show_gram(torch.tensor(components.T))

# Try on sklearn

## olivetti

In [ ]:
import logging

import matplotlib.pyplot as plt
from numpy.random import RandomState

from sklearn import cluster, decomposition
from sklearn.datasets import fetch_olivetti_faces

rng = RandomState(0)

# Display progress logs on stdout
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

faces, _ = fetch_olivetti_faces(return_X_y=True, shuffle=True, random_state=rng)
n_samples, n_features = faces.shape

# Global centering (focus on one feature, centering all samples)
faces_centered = faces - faces.mean(axis=0)

# Local centering (focus on one sample, centering all features)
faces_centered -= faces_centered.mean(axis=1).reshape(n_samples, -1)

print("Dataset consists of %d faces" % n_samples)

In [ ]:
n_row, n_col = 1,3
n_components = n_row * n_col
image_shape = (64, 64)


def plot_gallery(title, images, n_col=n_col, n_row=n_row, cmap=plt.cm.gray):
    fig, axs = plt.subplots(
        nrows=n_row,
        ncols=n_col,
        figsize=(2.0 * n_col, 2.3 * n_row),
        facecolor="white",
        constrained_layout=True,
    )
    fig.get_layout_engine().set(w_pad=0.01, h_pad=0.02, hspace=0, wspace=0)
    fig.set_edgecolor("black")
    fig.suptitle(title, size=16)
    for ax, vec in zip(axs.flat, images):
        vmax = max(vec.max(), -vec.min())
        im = ax.imshow(
            vec.reshape(image_shape),
            cmap=cmap,
            interpolation="nearest",
            vmin=-vmax,
            vmax=vmax,
        )
        ax.axis("off")

    fig.colorbar(im, ax=axs, orientation="horizontal", shrink=0.99, aspect=40, pad=0.01)
    plt.show()

In [ ]:
plot_gallery("Faces from dataset", faces_centered[:n_components])

In [ ]:
plot_gallery("our model", components[:n_components])

In [ ]:
# simplified linear model
# see what noise we can expect from this model, easy
model, codes, components, recon = train_baseline(faces, n_components, epochs=5000)

In [ ]:
base_noise_std = np.std(faces - recon)

In [ ]:
nmf_estimator = decomposition.NMF(n_components=n_components, tol=5e-3)
nmf_estimator.fit(faces)  # original non- negative dataset
plot_gallery("Non-negative components - NMF", nmf_estimator.components_[:n_components])

In [ ]:
for alpha in range(100,10_000, 250):
    model, codes, components, recon = train_grouping_autoencoder_fixed_sigma(
        faces, n_components, alpha, sigma_x=base_noise_std, sigma_0=1e-4, sigma_s=1, epochs=2500, lr=2e-3
    )
    plot_gallery(f"alpha: {alpha}", components[:n_components])

In [ ]:
# higher alphas seem a lot more predictable it seems, the requirements are much tighter

In [ ]:
show_gram(torch.tensor(components.T))

## Words decomp

In [ ]:
# Authors: The scikit-learn developers
# SPDX-License-Identifier: BSD-3-Clause

from time import time

import matplotlib.pyplot as plt

from sklearn.datasets import fetch_20newsgroups
from sklearn.decomposition import NMF, LatentDirichletAllocation, MiniBatchNMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

n_samples = 2000
n_features = 1000
n_components = 10
n_top_words = 20
batch_size = 128
init = "nndsvda"


def plot_top_words(components_, feature_names, n_top_words, title):
    fig, axes = plt.subplots(2, 5, figsize=(30, 15), sharex=True)
    axes = axes.flatten()
    for topic_idx, topic in enumerate(components_):
        top_features_ind = topic.argsort()[-n_top_words:]
        top_features = feature_names[top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7)
        ax.set_title(f"Topic {topic_idx + 1}", fontdict={"fontsize": 30})
        ax.tick_params(axis="both", which="major", labelsize=20)
        for i in "top right left".split():
            ax.spines[i].set_visible(False)
        fig.suptitle(title, fontsize=40)

    plt.subplots_adjust(top=0.90, bottom=0.05, wspace=0.90, hspace=0.3)
    plt.show()


# Load the 20 newsgroups dataset and vectorize it. We use a few heuristics
# to filter out useless terms early on: the posts are stripped of headers,
# footers and quoted replies, and common English words, words occurring in
# only one document or in at least 95% of the documents are removed.

print("Loading dataset...")
t0 = time()
data, _ = fetch_20newsgroups(
    shuffle=True,
    random_state=1,
    remove=("headers", "footers", "quotes"),
    return_X_y=True,
)
data_samples = data[:n_samples]
print("done in %0.3fs." % (time() - t0))

# Use tf-idf features for NMF.
print("Extracting tf-idf features for NMF...")
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.95, min_df=2, max_features=n_features, stop_words="english"
)
t0 = time()
tfidf = tfidf_vectorizer.fit_transform(data_samples)
print("done in %0.3fs." % (time() - t0))

# Use tf (raw term count) features for LDA.
print("Extracting tf features for LDA...")
tf_vectorizer = CountVectorizer(
    max_df=0.95, min_df=2, max_features=n_features, stop_words="english"
)
t0 = time()
tf = tf_vectorizer.fit_transform(data_samples)
print("done in %0.3fs." % (time() - t0))
print()

# Fit the NMF model
print(
    "Fitting the NMF model (Frobenius norm) with tf-idf features, "
    "n_samples=%d and n_features=%d..." % (n_samples, n_features)
)
t0 = time()
nmf = NMF(
    n_components=n_components,
    random_state=1,
    init=init,
    beta_loss="frobenius",
    alpha_W=0.00005,
    alpha_H=0.00005,
    l1_ratio=1,
).fit(tfidf)
print("done in %0.3fs." % (time() - t0))


tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
plot_top_words(
    nmf.components_, tfidf_feature_names, n_top_words, "Topics in NMF model (Frobenius norm)"
)

# Fit the NMF model
print(
    "\n" * 2,
    "Fitting the NMF model (generalized Kullback-Leibler "
    "divergence) with tf-idf features, n_samples=%d and n_features=%d..."
    % (n_samples, n_features),
)
t0 = time()
nmf = NMF(
    n_components=n_components,
    random_state=1,
    init=init,
    beta_loss="kullback-leibler",
    solver="mu",
    max_iter=1000,
    alpha_W=0.00005,
    alpha_H=0.00005,
    l1_ratio=0.5,
).fit(tfidf)
print("done in %0.3fs." % (time() - t0))

tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
plot_top_words(
    nmf.components_,
    tfidf_feature_names,
    n_top_words,
    "Topics in NMF model (generalized Kullback-Leibler divergence)",
)

# # Fit the MiniBatchNMF model
# print(
#     "\n" * 2,
#     "Fitting the MiniBatchNMF model (Frobenius norm) with tf-idf "
#     "features, n_samples=%d and n_features=%d, batch_size=%d..."
#     % (n_samples, n_features, batch_size),
# )
# t0 = time()
# mbnmf = MiniBatchNMF(
#     n_components=n_components,
#     random_state=1,
#     batch_size=batch_size,
#     init=init,
#     beta_loss="frobenius",
#     alpha_W=0.00005,
#     alpha_H=0.00005,
#     l1_ratio=0.5,
# ).fit(tfidf)
# print("done in %0.3fs." % (time() - t0))


# tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
# plot_top_words(
#     mbnmf.components_,
#     tfidf_feature_names,
#     n_top_words,
#     "Topics in MiniBatchNMF model (Frobenius norm)",
# )

# # Fit the MiniBatchNMF model
# print(
#     "\n" * 2,
#     "Fitting the MiniBatchNMF model (generalized Kullback-Leibler "
#     "divergence) with tf-idf features, n_samples=%d and n_features=%d, "
#     "batch_size=%d..." % (n_samples, n_features, batch_size),
# )
# t0 = time()
# mbnmf = MiniBatchNMF(
#     n_components=n_components,
#     random_state=1,
#     batch_size=batch_size,
#     init=init,
#     beta_loss="kullback-leibler",
#     alpha_W=0.00005,
#     alpha_H=0.00005,
#     l1_ratio=0.5,
# ).fit(tfidf)
# print("done in %0.3fs." % (time() - t0))

# tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
# plot_top_words(
#     mbnmf.components_,
#     tfidf_feature_names,
#     n_top_words,
#     "Topics in MiniBatchNMF model (generalized Kullback-Leibler divergence)",
# )

# print(
#     "\n" * 2,
#     "Fitting LDA models with tf features, n_samples=%d and n_features=%d..."
#     % (n_samples, n_features),
# )
# lda = LatentDirichletAllocation(
#     n_components=n_components,
#     max_iter=5,
#     learning_method="online",
#     learning_offset=50.0,
#     random_state=0,
# )
# t0 = time()
# lda.fit(tf)
# print("done in %0.3fs." % (time() - t0))

# tf_feature_names = tf_vectorizer.get_feature_names_out()
# plot_top_words(lda.components_, tf_feature_names, n_top_words, "Topics in LDA model")

In [ ]:
us = tfidf.toarray()

In [ ]:
vals = us[us !=0]

print(np.std(us), np.mean(us), np.percentile(us, 75), np.percentile(us, 90))
print(np.std(vals), np.mean(vals), np.percentile(vals, 75), np.percentile(vals, 90))

In [ ]:
# # in some seeds, recon loss does not descend in the beginnign
# # its useful to let it descend a bit in the beginning it seems, a few epochs maybe?
# # is sparse ness a problem with the other dataset?
# # if the data is already sparse, what happens to our weights regularisation?
# all_components, all_recons, all_codes = [], [], []

# code_maker_std = 1
# sigma_0_std = np.std(X) / 2
# sigma_noise = sigma_0_std / 100

# for seed in [10]:
#     print(f"#### seed run: {seed} #########")
#     torch.manual_seed(seed)
#     model, codes, components, recons = train_grouping_autoencoder_fixed_sigma(
#         X, N_COMPS, epochs=5000, sigma_0=sigma_0_std, sigma_s=code_maker_std, sigma_x=sigma_noise, max_alpha=10000, batch_size=256
#     )
#     all_components.append(components)
#     all_recons.append(recons)
#     all_codes.append(codes)

In [ ]:
std_pos, var_noise

In [ ]:
us = tfidf.toarray()
std_us = np.std(us)


std_pos = us[us != 0].std()

var_noise = 0.0001
var_w = 0.001
code_maker_std = 0.001

# the fundamental problem of fighting among each other, w and s, is giving me issues
# i want a uniform prior on s i think, but within a range. i can renormalize s every time maybe?
# the other way is to do coordiante descent. Do w and s in separate steps
torch.manual_seed(1)
model, codes, components, recons = train_grouping_autoencoder_fixed_sigma(
    us, n_components, epochs=2000, sigma_0=var_w, sigma_s=code_maker_std, sigma_x=var_noise, max_alpha=10_000, warmup_epochs=5, w_avg_num_seeds=1
)

In [ ]:
var_noise

In [ ]:
codes[0]

In [ ]:
# with relu, it might just be better to take a distribution which is generally negative
model, codes, components, recons = train_baseline(
    us, n_components, epochs=1000, sigma_x=1, sigma_s=1e-5
)

In [ ]:
codes[0]

In [ ]:
us[0][:10], recons[0][:10]

In [ ]:
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
plot_top_words(
    components,
    tfidf_feature_names,
    n_top_words,
    "Topics in our model",
)

In [ ]:
gram = show_gram(torch.tensor(components.T))
print(gram[5])

In [ ]:
gram